In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    monotonically_increasing_id,
    lit
)

sales = spark.table(
    "agentdb.silver.sales"
)

dim_product = spark.table(
    "agentdb.gold.dim_product"
).select(
    "product_key",
    "product_id"
)

dim_store = spark.table(
    "agentdb.gold.dim_store"
).select(
    "store_key",
    "store_id"
)

dim_calendar = spark.table(
    "agentdb.gold.dim_calendar"
).select(
    "calendar_key",
    "calendar_id"
)

fact_sales = (
    sales
    .join(
        dim_product,
        "product_id"
    )
    .join(
        dim_store,
        "store_id"
    )
    .join(
        dim_calendar,
        "calendar_id"
    )
    .withColumn(
        "sales_revenue",
        (sales["sales_qty"] * lit(0.0))
    )
    .withColumn(
        "created_at",
        current_timestamp()
    )
    .select(
        "product_key",
        "store_key",
        "calendar_key",
        "sales_qty",
        "sales_revenue",
        "created_at"
    )
)

(
    fact_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "agentdb.gold.fact_sales"
    )
)

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    monotonically_increasing_id,
    lit
)

store_inventory = spark.table(
    "agentdb.silver.store_inventory_snapshot"
)

dim_product = spark.table(
    "agentdb.gold.dim_product"
).select(
    "product_key",
    "product_id"
)

dim_store = spark.table(
    "agentdb.gold.dim_store"
).select(
    "store_key",
    "store_id"
)

dim_calendar = spark.table(
    "agentdb.gold.dim_calendar"
).select(
    "calendar_key",
    "date"
)

fact_inventory = (
    store_inventory
    .join(
        dim_product,
        "product_id"
    )
    .join(
        dim_store,
        "store_id"
    )
    .join(
        dim_calendar,
        store_inventory.snapshot_date
        == dim_calendar.date
    )
    .withColumn(
        "dc_key",
        lit(None)
    )
    .withColumn(
        "created_at",
        current_timestamp()
    )
    .select(
        "product_key",
        "store_key",
        "dc_key",
        "calendar_key",
        "inventory_qty",
        "days_of_cover",
        "created_at"
    )
)

(
    fact_inventory.write
    .mode("overwrite")
    .saveAsTable(
        "agentdb.gold.fact_inventory_snapshot"
    )
)

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    monotonically_increasing_id,
    datediff
)

po = spark.table(
    "agentdb.silver.purchase_order"
)

dim_product = spark.table(
    "agentdb.gold.dim_product"
).select(
    "product_key",
    "product_id"
)

dim_supplier = spark.table(
    "agentdb.gold.dim_supplier"
).select(
    "supplier_key",
    "supplier_id"
)

dim_dc = spark.table(
    "agentdb.gold.dim_distribution_center"
).select(
    "dc_key",
    "dc_id"
)

fact_po = (
    po
    .join(
        dim_product,
        "product_id"
    )
    .join(
        dim_supplier,
        "supplier_id"
    )
    .join(
        dim_dc,
        "dc_id"
    )
    .withColumn(
        "delay_days",
        datediff(
            po.actual_delivery_date,
            po.expected_delivery_date
        )
    )
    .withColumn(
        "created_at",
        current_timestamp()
    )
    .select(
        "product_key",
        "supplier_key",
        "dc_key",
        "ordered_qty",
        "received_qty",
        "lead_time_days",
        "delay_days",
        "order_date",
        "expected_delivery_date",
        "actual_delivery_date",
        "po_status",
        "created_at"
    )
)

(
    fact_po.write
    .mode("overwrite")
    .saveAsTable(
        "agentdb.gold.fact_purchase_orders"
    )
)

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    monotonically_increasing_id
)

disruptions = spark.table(
    "agentdb.silver.supplier_disruption"
)

dim_supplier = spark.table(
    "agentdb.gold.dim_supplier"
).select(
    "supplier_key",
    "supplier_id"
)

fact_disruptions = (
    disruptions
    .join(
        dim_supplier,
        "supplier_id"
    )
    .withColumn(
        "created_at",
        current_timestamp()
    )
    .select(
        "supplier_key",
        "disruption_type",
        "severity_score",
        "delay_days",
        "disruption_start_date",
        "disruption_end_date",
        "disruption_status",
        "created_at"
    )
)

(
    fact_disruptions.write
    .mode("overwrite")
    .saveAsTable(
        "agentdb.gold.fact_supplier_disruptions"
    )
)

In [0]:
agent_actions = spark.table(
    "agentdb.silver.agent_action_log"
)

(
    agent_actions
    .select(
        "action_id",
        "action_type",
        "entity_type",
        "entity_id",
        "action_status",
        "recommendation_timestamp",
        "resolved_timestamp",
        "created_at"
    )
    .write
    .mode("overwrite")
    .saveAsTable(
        "agentdb.gold.fact_agent_actions"
    )
)

In [0]:
tables = [
    "agentdb.gold.fact_sales",
    "agentdb.gold.fact_inventory_snapshot",
    "agentdb.gold.fact_purchase_orders",
    "agentdb.gold.fact_supplier_disruptions"
]

for table in tables:
    print(
        table,
        spark.table(table).count()
    )